In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)
import pandas as pd
import numpy as np # for random data generation
import matplotlib.pyplot as plt
import kagglehub
import os
import torch
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, LabelEncoder #import OneHotEncoder
from sklearn.model_selection import KFold ,StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm import tqdm
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from torch.utils.data import TensorDataset, DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize
from torch.optim import Adam, AdamW
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from collections import Counter
from sklearn.model_selection import StratifiedKFold


%pip install kagglehub catboost lightgbm tqdm -q
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q
from catboost import CatBoostClassifier


In [ ]:
# Task 1: Write your code here:
Exploring_Anonymized = os.path.join(path, '/kaggle/input/q3-ka-ai-2026/Q3_data.csv')
df = pd.read_csv(Exploring_Anonymized)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

cols = df.select_dtypes(include=['int64','float64']).columns
for col in [cols]:
    df[col] = df[col].fillna(df[col].median())


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:

In [ ]:
# Task 4: Write your code here:
numerical_cols =  df.select_dtypes(include=["number"]).columns.drop('Target')
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
feature_cols = [df.select_dtypes(include=['int64','float64']).columns.drop('Target')]
X = feature_cols
y = df['Target']


In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# using KFold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
 theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
 y_pred_proba = sigmoid(np.dot(X_test, theta))
 y_pred = (y_pred_proba >= 0.5).astype(int)

    # Calculate evaluation metrics
 accuracy = accuracy_score(y_test, y_pred)
 precision = precision_score(y_test, y_pred, zero_division=0)
 recall = recall_score(y_test, y_pred, zero_division=0)
 f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
  lr_accuracy.append(accuracy)
  lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
# Task 2,3,4,5: Write your code here: and im gonna use StratifiedKfold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  Precision: {np.mean(lr_precision):.4f}")
print(f"  Recall:    {np.mean(lr_recall):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Task 1: Write your code here:


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: